# Statistical Significance Test (Paired T-Test + Cohen's d + Bootstrap CI)

Produces Table V of the paper: paired t-test, Cohen's d, and 95% bootstrap confidence intervals comparing the BART baseline against BART + NLI reranking, on ROUGE-1/2/L and entailment, per test sample.

Recomputes ROUGE from scratch (CPU, `rouge_score`) and reuses the entailment scores already computed for the NLI-reranked predictions (`predictions_nli.jsonl`). **Section 2b recomputes entailment for the baseline predictions using the NLI model** — this is required because `predictions_baseline.jsonl`'s `entailment_score` field is a stored artifact of a separate pipeline bug (see the markdown note in Section 2b) and is always `0.0`, not a real score. This step benefits from a GPU accelerator if available but will fall back to CPU automatically (slower) if not.

Addresses reviewer feedback requesting paired significance testing (t-test, effect size, and confidence intervals) for the reported metrics.

## 1. Setup

In [ ]:
!pip install -q rouge-score scipy numpy

import json
import statistics
from pathlib import Path

import numpy as np
from scipy import stats
from rouge_score import rouge_scorer

SEED = 42
np.random.seed(SEED)

print("Setup OK")

## 2. Load Predictions

In [ ]:
# Update these two paths if your Kaggle input mount differs.
# These are the paths confirmed to exist from the "Output Notebook" dataset
# attached in 01_training.ipynb's Kaggle session (results/thesis_pipeline/outputs/).
PREDICTIONS_BASELINE_FILE = "/kaggle/input/datasets/madedwikibudilaksana/output-notebook/results/thesis_pipeline/outputs/predictions_baseline.jsonl"
PREDICTIONS_NLI_FILE = "/kaggle/input/datasets/madedwikibudilaksana/output-notebook/results/thesis_pipeline/outputs/predictions_nli.jsonl"

print("Baseline file exists:", Path(PREDICTIONS_BASELINE_FILE).exists())
print("NLI file exists:", Path(PREDICTIONS_NLI_FILE).exists())

def load_by_id(path):
    rows = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            rows[row["id"]] = row
    return rows

baseline_rows = load_by_id(PREDICTIONS_BASELINE_FILE)
nli_rows = load_by_id(PREDICTIONS_NLI_FILE)

print(f"Baseline predictions: {len(baseline_rows)}")
print(f"NLI predictions:      {len(nli_rows)}")

shared_ids = sorted(set(baseline_rows.keys()) & set(nli_rows.keys()))
print(f"Shared IDs (paired samples): {len(shared_ids)}")

only_baseline = set(baseline_rows.keys()) - set(nli_rows.keys())
only_nli = set(nli_rows.keys()) - set(baseline_rows.keys())
if only_baseline or only_nli:
    print(f"WARNING: {len(only_baseline)} IDs only in baseline, {len(only_nli)} IDs only in NLI file — these are excluded from the paired test.")

## 2b. Recompute Baseline Entailment (pipeline bug workaround)

**Bug found while validating this notebook's first run (2026-08-25):** in `01_training.ipynb`'s `generate_summaries()`, the baseline branch (`num_candidates == 1`) hardcodes `"entailment": 0.0, "contradiction": 0.0` instead of calling the NLI scorer, so `predictions_baseline.jsonl`'s `entailment_score` / `contradiction_score` fields are always `0.0` for every row — not real scores. The correct aggregate average (0.3100, matching Table IV) is computed separately in the notebook's evaluation cell but is never written back into the JSONL file. `predictions_nli.jsonl`'s `entailment_score` is unaffected and correct (it comes from the reranking branch, which does call the NLI scorer).

We work around this here by recomputing entailment/contradiction for the baseline predictions directly from the stored `document` and `generated_summary` fields, using the same NLI model as the rest of the pipeline. This does **not** require re-generating any summaries — only a forward pass through the NLI classifier per sample.

**Performance note (added after a run on CPU took ~10.2s/sample, i.e. ~31 hours projected for the full test set):** the cell below now batches inference (32 samples/batch on GPU, 8 on CPU) instead of scoring one sample at a time, and prints a timed runtime estimate from a single probe batch *before* committing to the full loop, so a too-slow run can be caught in seconds rather than hours. If the printed estimate is more than about an hour, stop the notebook, enable a GPU accelerator (Kaggle: Settings > Accelerator > GPU T4 x2 or P100), and re-run from the top — Kaggle's Commit-mode sessions start a fresh container per run, so there is no way to resume an in-progress CPU run after switching accelerators; it must restart from cell 1.

In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

NLI_MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32 if DEVICE == "cuda" else 8

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME).to(DEVICE).eval()
id2label = {int(k): v.lower() for k, v in nli_model.config.id2label.items()}
ent_idx = next(i for i, l in id2label.items() if "entail" in l)
con_idx = next(i for i, l in id2label.items() if "contrad" in l)

@torch.inference_mode()
def score_entailment_batch(premises, hypotheses):
    enc = nli_tokenizer(premises, hypotheses, truncation=True, max_length=512, padding=True, return_tensors="pt")
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    probs = torch.softmax(nli_model(**enc).logits, dim=-1)
    return probs[:, ent_idx].tolist(), probs[:, con_idx].tolist()

# --- Timed probe on one batch, to estimate total runtime BEFORE committing to the full loop.
# If this prints a multi-hour estimate on CPU, stop the notebook now, enable a GPU accelerator
# (Kaggle: Settings > Accelerator > GPU T4 x2 or P100), and re-run from the top instead of waiting.
_probe_n = min(BATCH_SIZE, len(shared_ids))
_probe_ids = shared_ids[:_probe_n]
_t0 = time.time()
_ = score_entailment_batch(
    [baseline_rows[i]["document"] for i in _probe_ids],
    [baseline_rows[i]["generated_summary"] for i in _probe_ids],
)
_probe_elapsed = time.time() - _t0
_est_total_s = (_probe_elapsed / _probe_n) * len(shared_ids)

print(f"Using device: {DEVICE} (batch size {BATCH_SIZE})")
print(f"Probe: {_probe_n} samples in {_probe_elapsed:.1f}s -> ~{_probe_elapsed / _probe_n:.3f}s/sample")
print(f"Estimated total time for {len(shared_ids)} samples: {_est_total_s / 60:.1f} min ({_est_total_s / 3600:.2f} hours)")
if _est_total_s > 3600:
    print("\n*** WARNING: estimated runtime exceeds 1 hour. ***")
    if DEVICE == "cpu":
        print("*** Strongly recommend stopping this notebook now, enabling a GPU accelerator ***")
        print("*** (Settings > Accelerator > GPU T4 x2 or P100) in Kaggle, and re-running from the top. ***")

baseline_entailment = {}
baseline_contradiction = {}
_start = time.time()
for i in range(0, len(shared_ids), BATCH_SIZE):
    batch_ids = shared_ids[i:i + BATCH_SIZE]
    premises = [baseline_rows[sid]["document"] for sid in batch_ids]
    hypotheses = [baseline_rows[sid]["generated_summary"] for sid in batch_ids]
    ents, cons = score_entailment_batch(premises, hypotheses)
    for sid, ent, con in zip(batch_ids, ents, cons):
        baseline_entailment[sid] = ent
        baseline_contradiction[sid] = con
    done = i + len(batch_ids)
    if done % (BATCH_SIZE * 10) < BATCH_SIZE or done == len(shared_ids):
        elapsed = time.time() - _start
        rate = elapsed / done
        remaining = rate * (len(shared_ids) - done)
        print(f"Baseline entailment recomputed: {done}/{len(shared_ids)} "
              f"({rate:.3f}s/sample, ~{remaining / 60:.1f} min remaining)")

recomputed_mean = statistics.mean(baseline_entailment.values())
print(f"\nRecomputed baseline avg entailment: {recomputed_mean:.4f} (should be close to 0.3100, per Table IV)")

## 3. Compute Per-Sample Scores

In [ ]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

per_sample = {
    "rouge1": {"baseline": [], "nli": []},
    "rouge2": {"baseline": [], "nli": []},
    "rougeL": {"baseline": [], "nli": []},
    "entailment": {"baseline": [], "nli": []},
}

for i, sample_id in enumerate(shared_ids):
    b = baseline_rows[sample_id]
    n = nli_rows[sample_id]

    b_scores = scorer.score(b["reference_summary"], b["generated_summary"])
    n_scores = scorer.score(n["reference_summary"], n["generated_summary"])

    per_sample["rouge1"]["baseline"].append(b_scores["rouge1"].fmeasure)
    per_sample["rouge1"]["nli"].append(n_scores["rouge1"].fmeasure)
    per_sample["rouge2"]["baseline"].append(b_scores["rouge2"].fmeasure)
    per_sample["rouge2"]["nli"].append(n_scores["rouge2"].fmeasure)
    per_sample["rougeL"]["baseline"].append(b_scores["rougeL"].fmeasure)
    per_sample["rougeL"]["nli"].append(n_scores["rougeL"].fmeasure)

    # baseline entailment comes from the Section 2b recomputation (predictions_baseline.jsonl's
    # stored entailment_score is always 0.0 due to the pipeline bug documented there).
    per_sample["entailment"]["baseline"].append(baseline_entailment[sample_id])
    per_sample["entailment"]["nli"].append(n["entailment_score"])

    if (i + 1) % 1000 == 0:
        print(f"Scored {i+1}/{len(shared_ids)}")

print(f"\nDone. Per-sample scores computed for {len(shared_ids)} paired samples.")

# Sanity check against the aggregate numbers already reported in Table IV / main_results.json
print("\nSanity check (should match Table IV / results/main_results.json):")
for metric in ["rouge1", "rouge2", "rougeL", "entailment"]:
    b_mean = statistics.mean(per_sample[metric]["baseline"])
    n_mean = statistics.mean(per_sample[metric]["nli"])
    print(f"  {metric:10s} baseline={b_mean:.4f}  nli={n_mean:.4f}")

## 4. Paired T-Test, Cohen's d, and Bootstrap CI

Cohen's d (paired, `d_z`) is computed as `mean(baseline - nli) / std(baseline - nli)`, matching the sign convention used elsewhere in this project: **positive d means baseline > NLI** (as for the ROUGE metrics, which slightly decrease under reranking), **negative d means NLI > baseline** (as for entailment, which increases under reranking). The direction is also printed explicitly per Reviewer 2's request (Minor Comment #3) rather than relying on the sign alone.

The 95% CI is a percentile bootstrap (10,000 resamples, seed=42) over the paired difference `nli - baseline`, reported in the *original* metric direction (i.e., positive means NLI is higher).

In [ ]:
def bootstrap_ci_mean_diff(baseline, nli, n_resamples=10000, ci=0.95, seed=SEED):
    rng = np.random.default_rng(seed)
    diffs = np.array(nli) - np.array(baseline)
    n = len(diffs)
    boot_means = np.empty(n_resamples)
    for i in range(n_resamples):
        sample_idx = rng.integers(0, n, size=n)
        boot_means[i] = diffs[sample_idx].mean()
    alpha = (1 - ci) / 2
    lower = float(np.quantile(boot_means, alpha))
    upper = float(np.quantile(boot_means, 1 - alpha))
    return lower, upper

def paired_cohens_d(baseline, nli):
    diffs = np.array(baseline) - np.array(nli)  # baseline - nli, matches project sign convention
    return float(diffs.mean() / diffs.std(ddof=1))

results = []
metric_labels = {"rouge1": "ROUGE-1", "rouge2": "ROUGE-2", "rougeL": "ROUGE-L", "entailment": "Entailment"}

for metric, label in metric_labels.items():
    baseline_scores = per_sample[metric]["baseline"]
    nli_scores = per_sample[metric]["nli"]

    baseline_mean = statistics.mean(baseline_scores)
    nli_mean = statistics.mean(nli_scores)

    t_stat, p_value = stats.ttest_rel(baseline_scores, nli_scores)
    d = paired_cohens_d(baseline_scores, nli_scores)
    ci_low, ci_high = bootstrap_ci_mean_diff(baseline_scores, nli_scores)

    direction = "NLI > baseline" if nli_mean > baseline_mean else "NLI < baseline"
    significant = p_value < 0.05

    result = {
        "metric": label,
        "baseline_mean": round(baseline_mean, 4),
        "nli_mean": round(nli_mean, 4),
        "t_statistic": round(float(t_stat), 3),
        "p_value": float(p_value),
        "significant_p<0.05": bool(significant),
        "cohens_d": round(d, 3),
        "direction": direction,
        "mean_diff_nli_minus_baseline": round(nli_mean - baseline_mean, 4),
        "bootstrap_95ci_low": round(ci_low, 4),
        "bootstrap_95ci_high": round(ci_high, 4),
        "n_samples": len(shared_ids),
    }
    results.append(result)

    p_str = f"{p_value:.2e}" if p_value >= 1e-300 else "< 1e-300"
    print(f"{label:10s} | baseline={baseline_mean:.4f} nli={nli_mean:.4f} | t={t_stat:8.3f} | p={p_str:>10s} | "
          f"sig={'Yes' if significant else 'No':3s} | d={d:7.3f} ({direction}) | "
          f"95% CI diff=[{ci_low:+.4f}, {ci_high:+.4f}]")

## 5. Save Results

In [ ]:
output_path = Path("./results/significance_test_results.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8") as f:
    json.dump({
        "n_paired_samples": len(shared_ids),
        "seed": SEED,
        "bootstrap_resamples": 10000,
        "cohens_d_convention": "paired d_z = mean(baseline - nli) / std(baseline - nli); positive d means baseline > NLI",
        "results": results,
    }, f, indent=2)

print(f"Saved to {output_path}")
print("\nCopy the table below into the paper's Table V:")
print(f"{'Metric':<10} | {'Baseline':>8} | {'NLI':>8} | {'T-stat':>8} | {'p-value':>10} | {'Sig':>3} | {'Cohens d':>9}")
print("-" * 70)
for r in results:
    p_str = f"{r['p_value']:.2e}" if r['p_value'] >= 1e-300 else "< 1e-300"
    print(f"{r['metric']:<10} | {r['baseline_mean']:>8.4f} | {r['nli_mean']:>8.4f} | {r['t_statistic']:>8.3f} | "
          f"{p_str:>10} | {'Yes' if r['significant_p<0.05'] else 'No':>3} | {r['cohens_d']:>9.3f}")